In [11]:
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

VBM_MIN, CBM_MAX = 1.23, 0.0
TOL = 0
REQUIRE_VBM = True


HERE = Path.cwd()
candidates = [HERE / "data",
              HERE,
              HERE / "nootebooks" / "data" / "data",
              HERE / "notebook" / "data"
              ]

DATA_DIR = next((p for p in candidates if (p / "EDGE_Cal.csv").exists()), None)
assert DATA_DIR is not None, "EDGE_Cal.csv not found. Run the extraction  notebook first."

IN_PATH = DATA_DIR / "EDGE_Cal.csv"
OUT_PATH = DATA_DIR / "BandEdge.csv"
FIG_DIR = DATA_DIR.parent / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", IN_PATH)
print("Output:", OUT_PATH)

Input: c:\Users\solom\Desktop\SemiconductorProject\notebook\data\data\EDGE_Cal.csv
Output: c:\Users\solom\Desktop\SemiconductorProject\notebook\data\data\BandEdge.csv


In [12]:
df = pd.read_csv(IN_PATH)
print("EDGE_Cal.csv shape", df.shape)
for  col in ["E_CBM", "E_VBM"]:
    assert col in df.columns, f"{col} column is missing from EDGE_Cal.csv"
df.head()



EDGE_Cal.csv shape (7223, 24)


,formula,elements,nelements,nsites,volume,density,crystal_system,space_group,formation_energy_per_atom,energy_above_hull,...,is_gap_direct,theoretical,stability_class,n_elements,mp_edge_gap_ok,chi_mulliken,E_CBM,E_VBM,E_CBM_vac,E_VBM_vac
0,Ac2S3,Ac_S,2,40,1118.407852007047,6.535149,Tetragonal,I-42d,-2.493064,0.0,...,False,True,stable,2,True,4.561109,-1.075341,1.197559,-3.424659,-5.697559
1,AcCrO3,Ac_Cr_O,3,5,61.36284504026082,8.848788,Cubic,Pm-3m,-3.138972,0.0,...,False,True,stable,3,True,5.394781,-0.106769,1.896331,-4.393231,-6.396331
2,AcGaO3,Ac_Ga_O,3,5,61.455078293307366,9.314495,Cubic,Pm-3m,-3.063253,0.0,...,False,True,stable,3,True,5.218015,-0.729935,2.165965,-3.770065,-6.665965
3,AcI3,Ac_I,2,8,465.6364161794002,4.334420,Hexagonal,P6_3/mmc,-1.801003,0.0,...,False,True,stable,2,True,5.451448,-0.342852,2.245748,-4.157148,-6.745748
4,Ag(BCl)6,Ag_B_Cl,3,104,1920.521817588254,2.666186,Cubic,Pa-3,-0.952834,0.0,...,True,False,stable,3,True,5.829141,-0.018609,2.676891,-4.481391,-7.176891


In [15]:
cbm_ok = df["E_CBM"] <= CBM_MAX + TOL
vbm_ok = df["E_VBM"] >= VBM_MIN - TOL
keep = (cbm_ok & vbm_ok) if REQUIRE_VBM else cbm_ok

edge_df = df[keep].copy().reset_index(drop=True)

print("Screening funnel")
print(f"    EDGE_Cal.csv                       : {len(df):,}")
print(f"    E_CBM <= {CBM_MAX + TOL:.2f} V                 : {int(cbm_ok.sum()):,}")
print(f"    E_VBM >= {VBM_MIN - TOL:.2f} V                 : {int(vbm_ok.sum()):,}")
print(f"    kept (BandEdge.csv)                            : {len(edge_df)} ({len(edge_df) / len(df):.1%})")
print(f"    removed in total                    : {len(df) - len(edge_df):,}")
edge_df.head()

Screening funnel
    EDGE_Cal.csv                       : 7,223
    E_CBM <= 0.00 V                 : 3,671
    E_VBM >= 1.23 V                 : 5,501
    kept (BandEdge.csv)                            : 1949 (27.0%)
    removed in total                    : 5,274


,formula,elements,nelements,nsites,volume,density,crystal_system,space_group,formation_energy_per_atom,energy_above_hull,...,is_gap_direct,theoretical,stability_class,n_elements,mp_edge_gap_ok,chi_mulliken,E_CBM,E_VBM,E_CBM_vac,E_VBM_vac
0,AcCrO3,Ac_Cr_O,3,5,61.36284504026082,8.848788,Cubic,Pm-3m,-3.138972,0.000000,...,False,True,stable,3,True,5.394781,-0.106769,1.896331,-4.393231,-6.396331
1,AcGaO3,Ac_Ga_O,3,5,61.455078293307366,9.314495,Cubic,Pm-3m,-3.063253,0.000000,...,False,True,stable,3,True,5.218015,-0.729935,2.165965,-3.770065,-6.665965
2,AcI3,Ac_I,2,8,465.6364161794002,4.334420,Hexagonal,P6_3/mmc,-1.801003,0.000000,...,False,True,stable,2,True,5.451448,-0.342852,2.245748,-4.157148,-6.745748
3,Ag(BCl)6,Ag_B_Cl,3,104,1920.521817588254,2.666186,Cubic,Pa-3,-0.952834,0.000000,...,True,False,stable,3,True,5.829141,-0.018609,2.676891,-4.481391,-7.176891
4,Ag2B8O13,Ag_B_O,3,92,999.27459655565,3.391399,Triclinic,P1,-2.499107,0.011548,...,False,True,near-stable,3,True,5.917450,-0.017050,2.851950,-4.482950,-7.351950


In [18]:
edge_df.to_csv(OUT_PATH, index=False)

summary = {
    "saved_at": datetime.now().isoformat(timespec="seconds"),
    "input_file": IN_PATH.name,
    "output_file": OUT_PATH.name,
    "edge_window_eV": [VBM_MIN, CBM_MAX],
    "row_in": int(len(df)),
    "rows_kept": int(len(edge_df)),
    

}
(DATA_DIR / f"{OUT_PATH.stem}_summary.json").write_text(json.dumps(summary, indent=2))
print("saved:", OUT_PATH)
print(json.dumps(summary, indent=2))

saved: c:\Users\solom\Desktop\SemiconductorProject\notebook\data\data\BandEdge.csv
{
  "saved_at": "2026-10-30T11:41:24",
  "input_file": "EDGE_Cal.csv",
  "output_file": "BandEdge.csv",
  "edge_window_eV": [
    1.23,
    0.0
  ],
  "row_in": 7223,
  "rows_kept": 1949
}
